# Lab 7: Image Smoothing and Convolutional Filters

**Computer Vision Course**

Welcome to your first encounter with **convolution** — one of the most fundamental operations in computer vision and deep learning.

**What you'll learn:**
- How convolution works
- The concept of kernels/filters
- Different smoothing strategies (simple vs Gaussian)
- Trade-offs between noise reduction and detail preservation

**What you'll build:**
- A simple averaging kernel
- Your own 2D convolution function
- A Gaussian kernel from the mathematical formula

**Connection to previous labs:**
- Lab 5: You learned brightness/contrast adjustments
- Lab 6: How we can implement linear operations with MLPs
- Lab 7 (today): You'll learn how to reduce noise while preserving details

**Looking ahead:**
This is the foundation for understanding how CNNs (Convolutional Neural Networks) work!

## Setup

In [ ]:
"""
Computer Vision Course - Lab 7: Image Smoothing and Convolution

This cell sets up the environment.
Works automatically for both local and Google Colab!
"""

import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

print("=" * 60)
print("Computer Vision - Lab 7: Image Smoothing")
print("=" * 60)

if IN_COLAB:
    print("\n🔵 Running on Google Colab")
    print("-" * 60)
    
    if not os.path.exists('computer-vision'):
        print("📥 Cloning repository...")
        !git clone https://github.com/mjck/computer-vision.git
        print("✓ Repository cloned successfully")
    else:
        !git -C computer-vision pull
        print("✓ Repository updated")
    
    %cd computer-vision/labs/lab07_smoothing
    print(f"✓ Current directory: {os.getcwd()}")
    
    sys.path.insert(0, '/content/computer-vision')
    print("✓ Python path configured")
    
    print("-" * 60)
    print("🟢 Colab setup complete!\n")
    
else:
    print("\n🟢 Running locally")
    print("-" * 60)
    print(f"✓ Current directory: {os.getcwd()}")
    
    repo_root = os.path.abspath('../..')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f"✓ Repository root: {repo_root}")
    
    print("-" * 60)
    print("🟢 Local setup complete!\n")

print("=" * 60)
print("✅ Environment ready!")
print("=" * 60)

## Import Libraries

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Import course utilities
try:
    from sdx import cv_imread, cv_imshow, cv_grayread
    print("✓ sdx module loaded")
except ImportError as e:
    print(f"❌ Could not import sdx: {e}")
    raise

print("✓ All imports successful")

## Choosing an Image

Image smoothing involves a fundamental trade-off:
> **The more noise you reduce, the more details you lose.**

To explore this trade-off, we have **18 test images** combining:
- **6 sources** (3 photos, 3 logos): Different detail levels and uniformity
- **3 noise severities**: Different amounts of corruption

This lets you see how your filters perform across different scenarios.

### Select Source

Available sources:
- **Photos:** `insper`, `informatica`, `harvard`  
- **Logos:** `atletica`, `smash`, `consulting`

Photos have varied details; logos have uniform regions and sharp edges.

In [ ]:
# Choose your source image
SOURCE = 'insper'  # Options: 'insper', 'informatica', 'harvard', 'atletica', 'smash', 'consulting'

print(f"✓ Selected source: {SOURCE}")

### Select Noise Severity

The number represents the approximate percentage of pixels corrupted by noise.

- `8` — Light noise (8% of pixels affected)
- `16` — Medium noise (16% of pixels affected)  
- `32` — Heavy noise (32% of pixels affected)

In [ ]:
# Choose noise severity
SEVERITY = 8  # Options: 8, 16, 32

print(f"✓ Selected severity: {SEVERITY}%")

## Loading and Displaying the Image

Like in Lab 5, we use `asfloat=True` to work with floating-point arrays.

In [ ]:
# Load the noisy image
input_image = cv_grayread(f'{SOURCE}-{SEVERITY}.png', asfloat=True)

print(f"Image shape: {input_image.shape}")
print(f"Data type: {input_image.dtype}")
print(f"Value range: [{input_image.min():.1f}, {input_image.max():.1f}]")

cv_imshow(input_image)
print(f"\nShowing: {SOURCE} with {SEVERITY}% noise")

---
## Activity 0: Understanding Simple Smoothing

Before you implement convolution yourself, let's understand what it does.

**Read the code below carefully.** Understand every line — you'll need this knowledge for the next activities.

In [ ]:
def simple_smooth(input, size):
    """
    Smooth an image by averaging pixels in a square neighborhood.
    
    Args:
        input: Input image (2D numpy array)
        size: Size of the averaging square (e.g., 3 for 3x3)
    
    Returns:
        Smoothed image (smaller than input due to boundary handling)
    
    How it works:
    - For each pixel, average all pixels in a 'size x size' window
    - Output is smaller because we can't center windows at the edges
    """
    height, width = input.shape
    
    radius = size // 2  # Half the window size (e.g., 3x3 → radius=1)
    
    total = size ** 2  # Number of pixels in the window (e.g., 3x3 → 9)
    
    # Output is smaller (can't process edges)
    output = np.empty((height - 2 * radius, width - 2 * radius))
    
    # Iterate over output pixels
    for oy, iy in enumerate(range(radius, height - radius)):
        for ox, ix in enumerate(range(radius, width - radius)):
            sum = 0
            
            # Sum all pixels in the window around (iy, ix)
            for dy in range(-radius, radius + 1):
                for dx in range(-radius, radius + 1):
                    sum += input[iy + dy, ix + dx]
            
            # Average by dividing by window size
            output[oy, ox] = sum / total
    
    return output

print("✓ simple_smooth() defined")

### Test Simple Smoothing

Use this as a **benchmark** for your implementations in the next activities.

In [ ]:
smoothed = simple_smooth(input_image, 3)

cv_imshow(smoothed)
print("Simple smoothing with 3x3 window")
print(f"Input size: {input_image.shape}")
print(f"Output size: {smoothed.shape}")
print("Notice: Output is 2 pixels smaller in each dimension (1 pixel lost on each edge)")

---
## Activity 1: Create a Simple Kernel

A **kernel** (also called a **filter**) is a small matrix of weights used in convolution.

For simple averaging, we want a kernel where:
- Every element has the same value
- All elements sum to 1 (to preserve brightness)

**For a 3×3 averaging kernel:**
```
[[1/9, 1/9, 1/9],
 [1/9, 1/9, 1/9],
 [1/9, 1/9, 1/9]]
```

**Your task:** Write a function that creates a `size × size` averaging kernel.

**Requirements:**
- Each element should be `1 / (size²)`
- Do **not** use loops — research NumPy array constructors
- Return a NumPy array

**Hints:**
- Look up `np.ones()` or `np.full()`
- Remember that `size ** 2` gives you the total number of elements

In [ ]:
def simple_kernel(size):
    """
    Create a simple averaging kernel.
    
    Args:
        size: Kernel size (e.g., 3 for 3x3 kernel)
    
    Returns:
        A size×size numpy array where each element is 1/(size²)
    
    TODO: Implement this function without loops
    """
    # ── Your code here ──────────────────────────────────────────────────────
    return np.empty((size, size))  # Replace this line
    # ────────────────────────────────────────────────────────────────────────

print("✓ Function defined. Test it below!")

### Test Your Kernel

If your function is correct, this should display:
```
array([[0.11111111, 0.11111111, 0.11111111],
       [0.11111111, 0.11111111, 0.11111111],
       [0.11111111, 0.11111111, 0.11111111]])
```

(which is `1/9 ≈ 0.111...` repeated 9 times)

In [ ]:
kernel = simple_kernel(3)
print("Your 3×3 kernel:")
print(kernel)

# Verify properties
print(f"\nKernel shape: {kernel.shape}")
print(f"Sum of all elements: {kernel.sum():.6f} (should be 1.0)")
print(f"Each element: {kernel[0,0]:.6f} (should be {1/9:.6f})")

---
## Activity 2: Implement Convolution

Now for the main event: implementing 2D convolution yourself!

**What is convolution?**
- Slide a kernel over an image
- At each position, multiply overlapping pixels by kernel weights
- Sum the products to get the output pixel

**Visual example (3×3 kernel on 5×5 image):**
```
Image:          Kernel:         Top-left Output pixel:
[a b c d e]     [k1 k2 k3]
[f g h i j]  *  [k4 k5 k6]  →  a×k1 + b×k2 + c×k3 +
[k l m n o]     [k7 k8 k9]     f×k4 + g×k5 + h×k6 +
[p q r s t]                    k×k7 + l×k8 + m×k9
[u v w x y]
```

**Your task:** Complete the `convolve()` function below.

**Requirements:**
- **DO use loops** — understanding the mechanics is important
- **DO NOT use** `scipy.signal.convolve2d` or similar library functions
- Follow the skeleton provided

In [ ]:
def convolve(input, kernel):
    """
    Perform 2D convolution of an image with a kernel.
    
    Args:
        input: Input image (2D numpy array)
        kernel: Convolution kernel (2D numpy array, assumed square)
    
    Returns:
        Convolved image (smaller than input)
    
    TODO: Implement the convolution operation using nested loops
    """
    height, width = input.shape
    
    # For simplicity, we only consider square kernels
    # (The code wouldn't change much for rectangular kernels)
    size, _ = kernel.shape
    
    radius = size // 2
    
    # Output is smaller (same as simple_smooth)
    output = np.empty((height - 2 * radius, width - 2 * radius))
    
    # ── Your code here ──────────────────────────────────────────────────────
    # Hint: You need 4 nested loops:
    # 1. Loop over output rows (oy, iy)
    # 2. Loop over output columns (ox, ix)
    # 3. Loop over kernel rows (dy)
    # 4. Loop over kernel columns (dx)
    #
    # At each position:
    # - Multiply input[iy+dy, ix+dx] by kernel[dy+radius, dx+radius]
    # - Sum all products
    # - Store in output[oy, ox]
    
    # ────────────────────────────────────────────────────────────────────────
    
    return output

print("✓ Function defined. Implement and test it below!")

### Test Your Convolution

Compare your output with the `simple_smooth` result from Activity 0. They should look identical!

In [ ]:
# Create a simple averaging kernel
kernel = simple_kernel(3)

# Apply your convolution
result = convolve(input_image, kernel)

cv_imshow(result)
print("Your convolution result with 3×3 averaging kernel")
print(f"Output shape: {result.shape}")
print("\n✓ If this looks the same as Activity 0, your convolution works!")

### Explore Different Noise Severities

Once your code works, test it on all three noise levels:

In [ ]:
print(f"Smoothing {SOURCE} with different noise severities:\n")

for severity in [8, 16, 32]:
    image = cv_grayread(f'images/{SOURCE}-{severity}.png', asfloat=True)
    result = convolve(image, kernel)
    cv_imshow(result)
    print(f"{severity}% noise")

print("\n🤔 Notice: Higher noise needs more smoothing, but that loses more detail!")

### Explore Different Sources

Test on all 6 sources with your selected severity:

In [ ]:
print(f"Smoothing all sources with {SEVERITY}% noise:\n")

sources = ['insper', 'informatica', 'harvard', 'atletica', 'smash', 'consulting']

for source in sources:
    image = cv_grayread(f'{source}-{SEVERITY}.png', asfloat=True)
    result = convolve(image, kernel)
    cv_imshow(result)
    print(f"{source}")

print("\n🤔 Notice: Logos (atletica, smash, consulting) lose sharp edges!")

---
## Challenge 1: Gaussian Kernel

Simple averaging treats all neighbors equally. **Gaussian smoothing** gives more weight to nearby pixels.

**The Gaussian formula:**

$$G(x, y) = \frac{1}{2\pi\sigma^2} e^{-\frac{x^2 + y^2}{2\sigma^2}}$$

Where:
- `(x, y)` are coordinates **relative to the kernel center**
- `σ` (sigma) controls the spread (larger σ = more blur)

**Your task:** Implement a function that creates a Gaussian kernel.

**Key challenge:** The origin `(0, 0)` should be at the **center** of the kernel, not the top-left.

**Example for 5×5 kernel:**
```
Coordinates:        Values (example with σ=1):
[-2,-2] [-1,-2] [0,-2] [1,-2] [2,-2]      [0.00  0.01  0.01  0.01  0.00]
[-2,-1] [-1,-1] [0,-1] [1,-1] [2,-1]      [0.01  0.04  0.06  0.04  0.01]
[-2, 0] [-1, 0] [0, 0] [1, 0] [2, 0]  →   [0.01  0.06  0.09  0.06  0.01]
[-2, 1] [-1, 1] [0, 1] [1, 1] [2, 1]      [0.01  0.04  0.06  0.04  0.01]
[-2, 2] [-1, 2] [0, 2] [1, 2] [2, 2]      [0.00  0.01  0.01  0.01  0.00]
           Center is at (0,0)                    Peak is at center
```

**Hints:**
- Create coordinate grids with `np.arange`
- Shift coordinates so center is at `(0, 0)`
- `np.exp()` for exponential
- `np.pi` for π

In [ ]:
def gaussian_kernel(size, sigma):
    """
    Create a Gaussian smoothing kernel.
    
    Args:
        size: Kernel size (must be odd, e.g., 3, 5, 7)
        sigma: Standard deviation of the Gaussian (controls spread)
    
    Returns:
        A size×size numpy array with Gaussian weights
    
    The formula:
        G(x,y) = (1 / (2π σ²)) × exp(-(x² + y²) / (2σ²))
    
    Where (x,y) are centered at the kernel's center.
    
    TODO: Implement this function
    """
    # ── Your code here ──────────────────────────────────────────────────────
    return np.empty((size, size))  # Replace this
    # ────────────────────────────────────────────────────────────────────────

print("✓ Function defined. Implement and test it below!")

### Test Your Gaussian Kernel

A correct Gaussian kernel should:
1. Be symmetric (same in all directions from center)
2. Have its peak at the center
3. Sum to 1.0 (preserve brightness)

In [ ]:
# Create a 5×5 Gaussian kernel with σ=1
gaussian = gaussian_kernel(5, 1.0)

print("Your 5×5 Gaussian kernel (σ=1.0):")
print(gaussian)

print(f"\nKernel shape: {gaussian.shape}")
print(f"Sum of weights: {gaussian.sum():.6f} (should be 1.0)")
print(f"Center value: {gaussian[2,2]:.6f}")
print(f"Corner value: {gaussian[0,0]:.6f}")
print(f"Peak at center: {gaussian[2,2] > gaussian[1,1]} (should be True)")

# Visualize it
plt.figure(figsize=(6, 5))
plt.imshow(gaussian, cmap='hot', interpolation='nearest')
plt.colorbar(label='Weight')
plt.title('5×5 Gaussian Kernel (σ=1.0)')
plt.show()

### Compare Simple vs Gaussian

Apply both kernels and compare:

In [ ]:
# Simple averaging kernel
simple = simple_kernel(3)
result_simple = convolve(input_image, simple)

# Gaussian kernel  
gaussian = gaussian_kernel(3, 1.0)
result_gaussian = convolve(input_image, gaussian)

print("Simple averaging (3×3):")
cv_imshow(result_simple)

print("\nGaussian smoothing (3×3, σ=1.0):")
cv_imshow(result_gaussian)

print("\n🤔 Can you see the difference? Gaussian is slightly less smooth.")

---
## Challenge 2: Critical Thinking

If your Gaussian implementation works correctly, you'll notice something odd:

**The image looks smoother, but also darker!**

### Your Task

**Question:** Why does the Gaussian-smoothed image appear darker?

Write your answer in the cell below. Think about:
- What does the Gaussian formula produce?
- How does it compare to the simple averaging kernel?
- What property should a smoothing kernel have to preserve brightness?

In [ ]:
# ── Your answer here ───────────────────────────────────────────────────────
# Why is the Gaussian-smoothed image darker?
#
# Your explanation:
# ...
#
#
# Hint: Check the sum of the Gaussian kernel values
# ────────────────────────────────────────────────────────────────────────────


### Investigate

Run this to examine the kernel properties:

In [ ]:
simple = simple_kernel(3)
gaussian = gaussian_kernel(3, 1.0)

print("Simple averaging kernel:")
print(simple)
print(f"Sum: {simple.sum():.6f}")

print("\nGaussian kernel:")
print(gaussian)
print(f"Sum: {gaussian.sum():.6f}")

print("\n💡 Aha! Look at the sums. What should they be?")

### Test Gaussian on Different Severities

In [ ]:
print(f"Gaussian smoothing on {SOURCE} with different noise levels:\n")

kernel = gaussian_kernel(3, 1.0)

for severity in [8, 16, 32]:
    image = cv_grayread(f'images/{SOURCE}-{severity}.png', asfloat=True)
    result = convolve(image, kernel)
    cv_imshow(result)
    print(f"{severity}% noise")

### Test Gaussian on Different Sources

In [ ]:
print(f"Gaussian smoothing on all sources ({SEVERITY}% noise):\n")

kernel = gaussian_kernel(3, 1.0)

for source in ['insper', 'informatica', 'harvard', 'atletica', 'smash', 'consulting']:
    image = cv_grayread(f'images/{source}-{SEVERITY}.png', asfloat=True)
    result = convolve(image, kernel)
    cv_imshow(result)
    print(f"{source}")

---
## OpenCV's Built-in Functions

Now that you've implemented convolution from scratch, you can appreciate what OpenCV does!

**OpenCV provides:**
- `cv2.blur()` — Simple averaging (what you built in Activity 1-2)
- `cv2.GaussianBlur()` — Gaussian smoothing (what you built in Challenge 1)

**Documentation:**
- [cv2.blur](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html#ga8c45db9afe636703801b0b2e440fce37)
- [cv2.GaussianBlur](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html#gaabe8c836e97159a9193fb0b11ac52cf1)

These are **much faster** than your implementation (optimized C++ code) and handle edge cases better.

In [ ]:
# Example: Using OpenCV's functions
import cv2

# Simple blur
cv_simple = cv2.blur(input_image, (3, 3))

# Gaussian blur (it handles the normalization correctly!)
cv_gaussian = cv2.GaussianBlur(input_image, (3, 3), sigmaX=1.0)

print("OpenCV simple blur:")
cv_imshow(cv_simple)

print("\nOpenCV Gaussian blur:")
cv_imshow(cv_gaussian)

print("\n✓ Notice: OpenCV's Gaussian is NOT darker (they normalize properly)")

---
## Summary

### What You Learned

1. **Convolution fundamentals**
   - Sliding a kernel over an image
   - Multiply-and-sum at each position
   - Output is smaller (edge handling)

2. **Different smoothing strategies**
   - Simple averaging: all neighbors equal
   - Gaussian: weight by distance from center

3. **Trade-offs**
   - More smoothing → less noise, less detail
   - Kernel size matters: 3×3 vs 5×5 vs 7×7
   - Kernel shape matters: simple vs Gaussian

4. **Implementation details**
   - Normalization is critical (sum to 1)
   - Coordinate systems (center vs corner)
   - Boundary handling (output size reduction)

### Key Insights

**Why convolution matters:**
- Fundamental operation in computer vision
- Core of Convolutional Neural Networks (CNNs)
- Same principle for edge detection, sharpening, etc.

**Noise vs detail trade-off:**
- You can't remove noise without affecting details
- Different images need different amounts of smoothing
- Photos vs logos behave differently

**Gaussian vs simple averaging:**
- Gaussian preserves edges better
- Simple averaging is faster to compute
- Both are "low-pass filters" (keep low frequencies, remove high)

### Looking Ahead

**Next topics:**
- Other types of kernels (edge detection, sharpening)
- Separable filters (efficiency trick)
- Frequency domain (Fourier transform)
- Convolutional Neural Networks (same operation, learned kernels!)